# Whisper 모델·양자화 자동 비교

고정 Validation 데이터에서 Whisper 모델 크기를 비교하고, 사람이 선택한 모델의 FP16·INT8-FP16 양자화를 비교한 뒤, 선택된 조합을 고정 Test에서 최종 평가합니다.

**보안:** 이 노트북은 카메라·마이크·패스키를 요청하지 않습니다. 공개 저장소와 공개 데이터만 사용하며, 실제 제조 음성은 승인·비식별 여부를 확인한 뒤 사용해야 합니다.

In [ ]:
GITHUB_REPO_URL = "https://github.com/Pronesis9758/aias-specialist-asr.git"
GITHUB_BRANCH = "codex/whisper-benchmark-quantization"  # PR 검증 후 main
PROJECT_DIR = "/content/AIAS"
DRIVE_ROOT = "/content/drive/MyDrive/AI_Specialist_ASR_Project"
BASE_CONFIG = "configs/colab_public_sample.yaml"
MODEL_MATRIX = "configs/benchmarks/whisper_models_colab.yaml"
QUANTIZATION_SPEC = "configs/quantization/whisper_quantization_colab.yaml"
BENCHMARK_ID = "public-whisper-model-benchmark-v1"
QUANTIZATION_ID = "public-whisper-quantization-v1"
# 원본 Hugging Face 모델까지 Drive에 보존하려면 True로 변경합니다.
# 약 15GB 이상의 추가 Drive 공간과 느린 첫 변환 I/O가 필요할 수 있습니다.
PERSIST_HF_SOURCE_CACHE = False

## 1. GPU 확인과 Google Drive 연결

변환이 끝난 CTranslate2 모델은 항상 Drive의 `models/`에 보존됩니다. `PERSIST_HF_SOURCE_CACHE=True`이면 변환 전 Hugging Face 원본도 Drive에 보존하지만, 기본값은 빠른 첫 실행과 Drive 용량 절약을 위해 `/content` 캐시입니다.

In [ ]:
!nvidia-smi
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

hf_cache_root = (
    Path(DRIVE_ROOT) / "cache/huggingface"
    if PERSIST_HF_SOURCE_CACHE
    else Path("/content/cache/huggingface")
)
hf_cache_root.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
print("Hugging Face source cache:", hf_cache_root)
print("Converted model cache:", Path(DRIVE_ROOT) / "models")

## 2. GitHub 코드 동기화

현재 저장소는 공개이므로 GitHub 토큰이 필요하지 않습니다.

In [ ]:
import os, subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            GITHUB_BRANCH,
            "--single-branch",
            GITHUB_REPO_URL,
            PROJECT_DIR,
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"], check=True
    )
os.chdir(PROJECT_DIR)
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. 의존성 설치

Whisper 원본 checkpoint를 CTranslate2 FP16·INT8 형식으로 변환하기 위해 Transformers 학습 extra를 함께 설치합니다.

In [ ]:
%pip uninstall -y torchao
%pip install -q -e ".[train]" "transformers>=4.46,<5" "peft>=0.14,<0.19"
import sys, torch, ctranslate2

print(
    {
        "python": sys.executable,
        "cuda": torch.cuda.is_available(),
        "gpu_compute_types": sorted(ctranslate2.get_supported_compute_types("cuda")),
    }
)

## 4. 공개 한국어 데이터 준비

공개 Zeroth-Korean 샘플을 Drive에 준비합니다. 실제 녹음으로 바꿀 때도 동일한 Manifest의 Validation/Test 분리를 유지해야 합니다.

In [ ]:
def run_aias(*args):
    command = [sys.executable, "-m", "aias_specialist.cli", *args]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)


run_aias("prepare-hf-dataset", "--config", BASE_CONFIG)
run_aias("doctor", "--config", BASE_CONFIG)

## 5. Whisper 모델 일괄 비교

Tiny, Base, Small, Medium, Large-v3, Turbo를 같은 Validation 데이터·FP16·beam size 5로 실행합니다. 완료된 모델은 Drive에 기록되어 재실행 시 건너뜁니다. `benchmark-models`가 모델 revision 고정까지 자동 수행하므로 별도 lock 명령은 필요하지 않습니다.

In [ ]:
run_aias("benchmark-models", "--matrix", MODEL_MATRIX)

In [ ]:
from pathlib import Path
import pandas as pd

benchmark_dir = Path(DRIVE_ROOT) / "artifacts/benchmarks" / BENCHMARK_ID
leaderboard = pd.read_csv(benchmark_dir / "benchmark_comparison.csv")
display(leaderboard.sort_values(["status", "rank"]))
print("Report:", benchmark_dir / "reports/benchmark_report.docx")

## 6. 모델 선택 — 사람이 확인하는 단계

`SELECTED_MODEL`을 비교표의 `member_id` 중 하나로 바꾸고 선택 이유를 작성합니다.

In [ ]:
SELECTED_MODEL = "small"
REVIEWER = "Pronesis9758"
MODEL_SELECTION_REASON = "Validation CER, 제조 용어 재현율, RTF와 GPU 메모리의 균형을 검토하여 선택"

run_aias(
    "select-model",
    "--benchmark-dir",
    str(benchmark_dir),
    "--model-id",
    SELECTED_MODEL,
    "--reviewer",
    REVIEWER,
    "--reason",
    MODEL_SELECTION_REASON,
)
model_selection = benchmark_dir / "model_selection.yaml"
print("Selection:", model_selection)

## 7. 선택 모델 양자화 비교

기본 실행은 FP16과 INT8-FP16입니다. FP32와 CPU INT8은 양자화 YAML에서 `enabled: true`로 바꾸면 추가할 수 있습니다.

In [ ]:
run_aias(
    "quantization-sweep",
    "--spec",
    QUANTIZATION_SPEC,
    "--selection",
    str(model_selection),
)

In [ ]:
quantization_dir = Path(DRIVE_ROOT) / "artifacts/quantization" / QUANTIZATION_ID
quantization_table = pd.read_csv(quantization_dir / "quantization_comparison.csv")
display(quantization_table.sort_values(["status", "rank"]))
print("Report:", quantization_dir / "reports/quantization_report.docx")

## 8. 양자화 단계 선택 — 사람이 확인하는 단계

FP16 대비 정확도 저하와 속도·메모리 이득을 확인한 뒤 선택합니다.

In [ ]:
SELECTED_VARIANT = "int8-float16"
QUANTIZATION_SELECTION_REASON = (
    "FP16 대비 CER와 제조 용어 재현율 손실이 허용 범위이며 메모리 효율이 좋아 선택"
)

run_aias(
    "select-quantization",
    "--quantization-dir",
    str(quantization_dir),
    "--variant-id",
    SELECTED_VARIANT,
    "--reviewer",
    REVIEWER,
    "--reason",
    QUANTIZATION_SELECTION_REASON,
)
quantization_selection = quantization_dir / "quantization_selection.yaml"

## 9. 고정 Test 최종평가

모델과 양자화 선택이 끝난 이후에만 Test split을 실행합니다.

In [ ]:
run_aias(
    "finalize-evaluation",
    "--selection",
    str(quantization_selection),
    "--config",
    BASE_CONFIG,
)
print("Final result:", quantization_dir / "final_test_result.json")

## 산출물

- 모델 비교 CSV·그래프·Word 보고서
- 모델 선택자와 선택 이유
- 양자화 비교 CSV·그래프·Word 보고서
- 양자화 선택자와 선택 이유
- 고정 Test 최종 예측·지표·Word 보고서
- 모든 독립 run과 SQLite 실험 이력

최종 보고서 결론과 실제 제조 데이터의 정답·개인정보 승인은 반드시 사람이 검수합니다.